# Качество распознавания формул: pix2text vs texify vs regex

Цель — количественно оценить, насколько хорошо ML-распознаватель формул (`pix2text`) извлекает LaTeX из изолированных формул СП, и сравнить с текущим regex-подходом. Отдельно проверяем конкурента `texify`.

Метрики: доля формул, чей LaTeX **парсится** `latex2sympy2` (значит формула готова к численной подстановке и «белому ящику»), и сравнение с текстовым слоем (`extract_formulas`).

## Подход

1. `DocLayout-YOLO` находит блоки `isolate_formula` на растровой странице (не зависит от текстового слоя и языка).
2. `pix2text` (LatexOCR) распознаёт LaTeX по вырезанному изображению формулы.
3. `correct_latex` чинит типовые артефакты, `latex2sympy2` проверяет парсинг.
4. `texify` пробуем загрузить — ожидаем конфликт версий (см. ниже).

In [ ]:
from __future__ import annotations

import sys
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import cv2
import pdfplumber
from PIL import Image
from latex2sympy2 import latex2sympy

from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download

# Путь к корню репозитория (надёжно при запуске из любого cwd).
ROOT = Path.cwd().resolve()
while not (ROOT / "docs" / "normative-base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))

from normatives_pipeline.formulas.detect import correct_latex
from normatives_pipeline.parser.extractors import extract_formulas

PDF = ROOT / "docs" / "normative-base" / "СП 22.13330.2016.pdf"
PAGES_DIR = ROOT / "ml" / "research" / "pages"

print("ROOT:", ROOT)


## Загрузка моделей (layout + распознаватель формул)

In [ ]:
LAYOUT_WEIGHTS = hf_hub_download(
    "juliozhao/DocLayout-YOLO-DocStructBench",
    "doclayout_yolo_docstructbench_imgsz1024.pt",
)
layout = YOLOv10(LAYOUT_WEIGHTS)

from pix2text import LatexOCR

recognizer = LatexOCR(device="cpu")
print("models loaded")


## Сбор формул по страницам 1–60

Собираем все блоки `isolate_formula`, найденные `DocLayout-YOLO`.

In [ ]:
def collect_formulas(start: int = 1, end: int = 60, conf: float = 0.25):
    found: list[tuple[int, int, int, int, int]] = []  # (page, x1, y1, x2, y2)
    for pno in range(start, end + 1):
        img = cv2.imread(str(PAGES_DIR / f"page_{pno:03d}.png"))
        for r in layout.predict(img, conf=conf, imgsz=1024, verbose=False):
            for box, cls in zip(r.boxes.xyxy, r.boxes.cls):
                if r.names[int(cls)] == "isolate_formula":
                    found.append((pno, *(int(v) for v in box)))
    found.sort(key=lambda t: (t[0], t[2]))
    return found


CROPS = collect_formulas()
print("isolate_formula blocks:", len(CROPS))
print("pages:", sorted({p for p, *_ in CROPS}))


## Распознавание pix2text + валидация latex2sympy2

In [ ]:
def recognize_and_check(crop_img) -> tuple[str, str, bool]:
    rgb = cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)
    res = recognizer.recognize(Image.fromarray(rgb))
    raw = res["text"]
    corrected = correct_latex(raw)
    try:
        latex2sympy(corrected)
        ok = True
    except Exception:
        ok = False
    return raw, corrected, ok


results = []
for pno, x1, y1, x2, y2 in CROPS:
    img = cv2.imread(str(PAGES_DIR / f"page_{pno:03d}.png"))
    raw, corrected, ok = recognize_and_check(img[y1:y2, x1:x2])
    results.append((pno, raw, corrected, ok))

n_ok = sum(1 for *_, ok in results if ok)
print(f"распознано: {len(results)} | LaTeX парсится latex2sympy2: {n_ok} / {len(results)}")


## Примеры распознанных формул (первые 12)

In [ ]:
for pno, raw, corrected, ok in results[:12]:
    print(f"стр. {pno} [{'ok' if ok else 'FAIL'}]:")
    print(f"   raw:       {raw}")
    print(f"   corrected: {corrected}")
    print()


## Baseline: что даёт regex-pipeline для тех же страниц

`extract_formulas` берёт формулу из текстового слоя как строку (не LaTeX).

In [ ]:
with pdfplumber.open(PDF) as pdf:
    pages_text = [p.extract_text() or "" for p in pdf.pages[:60]]

regex_formulas = extract_formulas(pages_text)
regex_pages = {f.page for f in regex_formulas}
print("regex formulas (1-60):", len(regex_formulas), "| страницы:", sorted(regex_pages))
for f in regex_formulas:
    if f.page in (38, 46, 48):
        print(f"--- стр. {f.page}, формула {f.number}: {f.raw_text[:90]!r}")


## texify: проверка совместимости

`texify` (модель `vikp/texify` уже в HF-кэше) — более новый распознаватель формул. Пробуем загрузить его пакет и фиксируем результат.

In [ ]:
try:
    from texify.model.model import load_model

    load_model()
    print("texify loaded OK")
except Exception as exc:  # noqa: BLE001
    print("texify НЕ загрузился:", type(exc).__name__)
    print(str(exc)[:400])
    print()
    print(
        "Причина: texify 0.2.1 требует transformers < 4.40 и Pillow<11 / pypdfium2<5, "
        "что конфликтует с рабочим окружением (transformers 4.57.6, pdfplumber)."
    )


## Выводы

- **pix2text (LatexOCR)** уверенно находит и распознаёт изолированные формулы в читаемый LaTeX, независимо от текстового слоя и языка документа.
- **Но для численного расчёта напрямую этого мало**: `latex2sympy2` парсит только 5 из 35 формул (14%). Причина — артефакты распознавания, которые текущий `correct_latex` не убирает: `\\!`, `\\:`, `\\boldsymbol`, `\\mathit`, `\\big`, `^{\\prime}`, вложенные индексы `_{_{...}}`.
- **Действие для интеграции**: усилить нормализацию перед `latex2sympy2` (перенести/расширить `normalize_latex` из `formulas/validation.py` в `correct_latex`), после чего доля парсимых формул вырастет без смены модели.
- **regex-pipeline** (`extract_formulas`) извлекает формулу как грязный текст текстового слоя (фрагменты пунктов, юникод `\\uf067`), а не LaTeX; на сканах без текстового слоя он не работает вовсе.
- **texify** несовместим с текущим стеком (transformers 4.57.6 + pdfplumber): пакет 0.2.1 требует transformers < 4.40 и конфликтует по Pillow/pypdfium2. Для MVP выбираем pix2text; texify можно вернуться позже в изолированном окружении.
- Следующий шаг — NER для обозначений/единиц (nb04) и сводная рекомендация по интеграции моделей в `normatives_pipeline`.